In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import tools_condition, ToolNode
from langchain_tavily import TavilySearch
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage

from typing_extensions import Annotated, TypedDict
load_dotenv()

### Define the LLM Model
llm = init_chat_model("groq:openai/gpt-oss-20b")

### Tools
searchTool = TavilySearch(max_results = 2)

@tool
def calculations(old_value: float, new_value: float) -> str:
    """Calculate percentage growth or decline between two values."""

    if old_value == 0:
        return "Cannot calculate percentage change because the old value is zero."

    percentage = ((new_value - old_value) / old_value) * 100

    if percentage > 0:
        return f"Growth: {percentage:.2f}%"
    elif percentage < 0:
        return f"Decline: {abs(percentage):.2f}%"
    else:
        return "No change: 0%"

### combine the tools
tools = [searchTool, calculations]

### llm with tools
llmWithTools = llm.bind_tools(tools)


### State
class State(TypedDict):
    messages: Annotated[list, add_messages]


### Nodes
def Analyzer(state: State):
    response = llm.invoke([
        SystemMessage(content="""
        Analyze the user's query.
        Determine whether web research or calculations are required.
        """),
        *state["messages"]
    ])

    return {"messages": [response]}


def agent(state: State):
    return llmWithTools.invoke(state["messages"])


def summarizer(state: State):
    response = llm.invoke([
            SystemMessage(content="""
            You have to summarize the response and return the final report
            """),
            *state["messages"]
        ])
    
    return {"messages": [response]}


### Initialize the Graph
graph_builder = StateGraph(State)


graph_builder.add_node("Analyzer", Analyzer)
graph_builder.add_node("agent", agent)
graph_builder.add_node("summarizer", summarizer)
graph_builder.add_node("tools", ToolNode(tools))



graph_builder.add_edge(START, "Analyzer")
graph_builder.add_edge("Analyzer", "agent")
graph_builder.add_conditional_edges("agent", tools_condition,
    {
        "tools": "tools",
        END: "summarizer"
    })
graph_builder.add_edge("tools", "agent")
graph_builder.add_edge("summarizer", END)

compiledGraph = graph_builder.compile()
response = compiledGraph.invoke({"messages": [HumanMessage(content="Research the latest developments in LangGraph and compare it with CrewAI.")]})
print(response["messages"][-1])
    



BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "browser.run", "arguments": {"id":"browser.run","params":{"command":"search","query":"LangGraph latest developments 2024","top_k":10,"source":"news"}}\n}'}}